In [2]:
import pandas as pd

url = "https://www.data.gouv.fr/api/1/datasets/r/bcd7d584-89c3-46b6-a164-4582eb2d43b5"
df = pd.read_csv(url, sep=";", encoding="utf-8")

print(df.shape)
df.head()

(3237, 8)


,librairie,assortiment,label,Commune,code_commune,Code_postal,Departement,Region
0,BLANCHE NEIGE,Généraliste,NaN,AMBERIEU EN BUGEY,01004,1500,AIN,AUVERGNE RHONE ALPES
1,ENTRE PARENTHESES,Généraliste,NaN,BELLEY,01034,1300,AIN,AUVERGNE RHONE ALPES
2,LIBRAIRIE ZANNINI,Généraliste,NaN,BOURG EN BRESSE,01053,1000,AIN,AUVERGNE RHONE ALPES
3,MONTBARBON,Généraliste,LIR,BOURG EN BRESSE,01053,1000,AIN,AUVERGNE RHONE ALPES
4,ZANNINI,Généraliste,NaN,BOURG EN BRESSE,01053,1000,AIN,AUVERGNE RHONE ALPES


In [3]:
# Préparation des variables (noms de colonnes et types)
df = df.copy()

# Le fichier contient un espace final dans "code_commune "
df = df.rename(columns={"code_commune ": "code_commune"})

# Uniformiser les codes
for col in ["code_commune", "Code_postal"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .str.zfill(5)
    )

df[["librairie", "assortiment", "Commune", "code_commune", "Code_postal"]].head()

,librairie,assortiment,Commune,code_commune,Code_postal
0,BLANCHE NEIGE,Généraliste,AMBERIEU EN BUGEY,01004,01500
1,ENTRE PARENTHESES,Généraliste,BELLEY,01034,01300
2,LIBRAIRIE ZANNINI,Généraliste,BOURG EN BRESSE,01053,01000
3,MONTBARBON,Généraliste,BOURG EN BRESSE,01053,01000
4,ZANNINI,Généraliste,BOURG EN BRESSE,01053,01000


In [4]:
import requests

# Récupération de toutes les communes avec leur centroïde (lat/lon)
url_communes = (
    "https://geo.api.gouv.fr/communes"
    "?fields=code,nom,centre,codesPostaux"
    "&format=json"
    "&geometry=centre"
)

communes = requests.get(url_communes, timeout=60).json()
df_communes = pd.DataFrame(communes)

# Coordonnées depuis "centre.coordinates" -> [lon, lat]
df_communes["lon"] = df_communes["centre"].apply(
    lambda x: x.get("coordinates", [None, None])[0] if isinstance(x, dict) else None
)
df_communes["lat"] = df_communes["centre"].apply(
    lambda x: x.get("coordinates", [None, None])[1] if isinstance(x, dict) else None
)

df_communes = df_communes[["code", "nom", "lon", "lat"]].rename(
    columns={"code": "code_commune", "nom": "nom_commune_ref"}
)

# Merge sur code INSEE commune
df_map = df.merge(df_communes, on="code_commune", how="left")

print("Lignes:", len(df_map))
print("Coordonnées manquantes:", df_map[["lat", "lon"]].isna().any(axis=1).sum())
df_map.head()

Lignes: 3237
Coordonnées manquantes: 0


,librairie,assortiment,label,Commune,code_commune,Code_postal,Departement,Region,nom_commune_ref,lon,lat
0,BLANCHE NEIGE,Généraliste,NaN,AMBERIEU EN BUGEY,01004,01500,AIN,AUVERGNE RHONE ALPES,Ambérieu-en-Bugey,5.3706,45.9575
1,ENTRE PARENTHESES,Généraliste,NaN,BELLEY,01034,01300,AIN,AUVERGNE RHONE ALPES,Belley,5.6927,45.7430
2,LIBRAIRIE ZANNINI,Généraliste,NaN,BOURG EN BRESSE,01053,01000,AIN,AUVERGNE RHONE ALPES,Bourg-en-Bresse,5.2469,46.2027
3,MONTBARBON,Généraliste,LIR,BOURG EN BRESSE,01053,01000,AIN,AUVERGNE RHONE ALPES,Bourg-en-Bresse,5.2469,46.2027
4,ZANNINI,Généraliste,NaN,BOURG EN BRESSE,01053,01000,AIN,AUVERGNE RHONE ALPES,Bourg-en-Bresse,5.2469,46.2027


In [5]:
# Installer folium si nécessaire
try:
    import folium
    from folium.plugins import MarkerCluster
except ModuleNotFoundError:
    %pip install folium
    import folium
    from folium.plugins import MarkerCluster

In [6]:
# Carte 1 : toutes les librairies (points)
map_all = folium.Map(location=[46.6, 2.4], zoom_start=6, tiles="CartoDB positron")
cluster_all = MarkerCluster(name="Librairies").add_to(map_all)

for _, r in df_map.dropna(subset=["lat", "lon"]).iterrows():
    popup = (
        f"<b>{r['librairie']}</b><br>"
        f"Commune: {r['Commune']} ({r['Code_postal']})<br>"
        f"Assortiment: {r['assortiment']}"
    )
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=3,
        color="#1f77b4",
        fill=True,
        fill_opacity=0.7,
        popup=popup,
    ).add_to(cluster_all)

folium.LayerControl().add_to(map_all)
map_all

In [8]:
# Carte 3 : un point par ville (Commune + Code postal), taille = nb de librairies
city_points = (
    df_map.dropna(subset=["lat", "lon"])
    .groupby(["Commune", "code_commune", "Code_postal", "lat", "lon"], as_index=False)
    .agg(nb_librairies=("librairie", "count"))
)

map_city = folium.Map(location=[46.6, 2.4], zoom_start=6, tiles="CartoDB positron")

for _, r in city_points.iterrows():
    radius = min(2 + r["nb_librairies"] * 0.7, 18)
    popup = (
        f"<b>{r['Commune']}</b><br>"
        f"Code commune: {r['code_commune']}<br>"
        f"Code postal: {r['Code_postal']}<br>"
        f"Nombre de librairies: {r['nb_librairies']}"
    )
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=radius,
        color="#2ca25f",
        fill=True,
        fill_color="#2ca25f",
        fill_opacity=0.6,
        popup=popup,
    ).add_to(map_city)

map_city